In [1]:
!pip install -U langchain==1.4.6 faiss-cpu==1.15.1 cohere==7.2.0 langchain-community==0.4.2 rank-bm25==0.2.2 sentence-transformers==6.1.0
!pip install llama-cpp-python==0.2.78  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.7/370.7 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5

In [2]:
ZHIPU_APIKEY = "你的智谱APIKey"

# 零、总述

Search(搜索) 是最早广泛应用于行业的大语言模型应用之一。在具有里程碑意义发布的论文 BERT【Pre-training of deep bidirectional transformers for language understanding】（2018年发布） 后的几个月, Google 使用了此方法来支持搜索，并且 Google 表示：它是 Search 历史上的一大里程碑式的飞跃。与此同时，Microsoft Bing 也不甘落后，它表示：“从今年 4 月开始，我们使用大小 transform 模型，为 Bing 用户带来了过去一年中幅度最大的质量提升。”

它们带来的这种能力被称为 **语义搜索（Semantic Search）**，它使搜索能够基于文本的含义进行，而不仅仅依靠关键词匹配。

与此同时，文本生成模型的快速普及使许多用户开始向模型提问，并期待模型能够给出事实性的答案。虽然这些模型能够流畅且自信地回答问题，但它们给出的答案并不总是正确的，也不一定是最新的。这个问题被称为**幻觉（Hallucinations）**。目前减少幻觉的一种主要方法，是构建能够检索相关信息的系统，并将检索到的信息提供给 LLM，从而帮助模型生成更符合事实的答案。终止方法被称为 **RAG（Retrieval-Augmented Generation）**

下面介绍三个概念：Dense Retrieval，Reranking, 和 RAG，这些概念不仅仅应用在 LLM 的搜索域中。



1. **Dense Retrieval(Dense 检索)**

Dense 检索的系统依赖于 Embedding 的概念，它会将搜索的问题转换成检索离此问题最近邻的搜索上。如下图所示：

<center>
<img src="./resources/dense_retrieval.png">
</center>

2. **Reranking(重排)**

搜索通常包含多步，重排序的语言模型通常是这些步骤之一，它的目标是计算代表检索出的文档与查询的相关性的一个分数，之后它就会按照这个分数进行重排。如下图所示：

<center>
<img src="./resources/reranking.png">
</center>


3. **RAG**

生成式搜索（Generative Search）是一个更广泛的一类子集，我们通常叫他 RAG。文本生成系统会包含搜索能力以减少幻觉，提升事实的准确性，还可以让生成模型基于指定的数据集生成内容。RAG 的流程如下图所示：

<center>
<img src="./resources/rag.png">
</center>


# 一、使用语言模型进行语义搜索

## 1.1 Dense 检索

Embedding 会将文本转换成空间的某个点，它们的距离越近，表示两个文本越相似。想象这样一个场景，用户在问题的时候，我们会将问题嵌入到与文本库相同的空间中，之后，我们去找与它最近的几个文档即可。如下图所示：query 代表问题，text n 代表文档

<center>
<img src="./resources/relevant.png">
</center>

可以看到，“text 2” 是问题 query 最好的回答，接下来是 "text 1"。那么这就会出现两个问题：

- text 3 是否应该作为结果返回呢？这个取决于你。有时，我们会设定一个很大的**阈值**来将不太相关的内容纳入进来
- 问题 query 和最好的回答语义相似么？并不总是这样。这就是为什么语言模型需要在 QA 对上反复的训练来变得更优秀

### 1.1.1 示例

让我们基于智谱官网的模型介绍，来探索关于 GLM-5.3 的内容

我们分为以下四步：

1. 我们会将相关文档切成每一句来让文本变得可搜索
2. 将每一句嵌入到向量空间中
3. 构造搜索索引
4. 搜索并查看结果

#### 1. Getting the text archive and chunking it

我们只使用其中的一部分，完整的文章在 [GLM 5.3介绍](https://docs.bigmodel.cn/cn/guide/models/text/glm-5.3)

In [3]:
text = """
在 GLM-5.3 的训练中，我们进一步推进了任务环境的规模化建设，使训练任务不再只是传统的编程题，而是更接近专家在真实工作中承担的完整专业工作单元。这些环境覆盖了更广泛的生产级工作流，任务设计也更加贴近工程与研究工作的实际开展方式，其中部分任务即使由经验丰富的工程师完成，也需要数天时间。
例如，在一项机器学习基础设施任务中，模型可能会获得与工程师相同的工作环境，包括计算集群、存储系统、内部文档、代码库和实验结果等资源。模型需要定位训练技术栈中的性能瓶颈，实施优化方案，运行实验，并在保证结果正确性的前提下，实现可量化的端到端加速。通过在此类高复杂度环境中训练，推动模型逐步具备端到端负责并完成复杂工作的能力，而不再依赖用户拆解问题并逐步监督执行。
随着智能体能力提升，后训练规模化的主要难点也逐渐从模型本身转移到任务环境。一个真正有价值的任务环境必须可执行、可验证，并贴近真实的专业工作；同时，我们需要的不是少量人工构建的环境，而是大规模、多样化的任务环境。为实现这一过程的规模化，我们构建了能够端到端合成任务环境的流水线，并针对部分任务进一步自动生成强化学习所需的奖励信号。研究智能体从真实工作中收集任务模式，并将其转化为可运行的长程任务环境，其中包含多步骤依赖关系和隐藏状态；随后，评审智能体会实际尝试完成每项任务，以验证任务是否确实可解。验证器在不访问参考解法的前提下生成；与此同时，系统会利用求解轨迹识别并消除奖励捷径。只有同时通过 Oracle、空操作和未解决状态检查的验证器，其生成的二值奖励信号才足够可靠，可直接用于模型训练。
GLM-5.3 延续了 GLM-5.2 中引入的强化学习策略，包括结合上下文压缩机制的 SAO，使能力增益能够在长程任务中持续体现，而非仅局限于短程任务。这些提升同时反映在编程任务和通用智能体任务中：GLM-5.3 在 Terminal-Bench 3.0 上的得分由 4.6 提升至 28.3，在 DeepSWE v1.1 上由 46.2 提升至 66.9，在 Agents’ Last Exam 上由 23.8 提升至 28.5。目前，这些流水线仍需要较多人工参与。进一步提升任务环境生成与验证过程的自主化程度，是我们下一阶段的重点方向之一。
除公开基准测试外，我们还推出了内部基准 Z.ai Code Bench，用于在贴近真实用户场景的条件下评估编程智能体。该基准覆盖多种任务类别，并将智能体置于复杂的本地开发环境中。在不同推理强度档位下，我们从两个维度评估智能体：一是端到端任务完成率，二是细粒度检查项准确率。作为一项非公开基准，Z.ai Code Bench 还能降低公开测试集污染带来的风险，从而更准确地反映模型在真实用户场景中的使用体验。"""

# 使用句号将文本转成 list
texts = text.split('。')

# 去空格并加换行
texts = [t.strip(' \n') for t in texts if isinstance(t, str) and t.strip()]

In [4]:
print(type(texts))
print(len(texts))
print(repr(texts[:3]))

<class 'list'>
19
['在 GLM-5.3 的训练中，我们进一步推进了任务环境的规模化建设，使训练任务不再只是传统的编程题，而是更接近专家在真实工作中承担的完整专业工作单元', '这些环境覆盖了更广泛的生产级工作流，任务设计也更加贴近工程与研究工作的实际开展方式，其中部分任务即使由经验丰富的工程师完成，也需要数天时间', '例如，在一项机器学习基础设施任务中，模型可能会获得与工程师相同的工作环境，包括计算集群、存储系统、内部文档、代码库和实验结果等资源']


In [5]:
texts

['在 GLM-5.3 的训练中，我们进一步推进了任务环境的规模化建设，使训练任务不再只是传统的编程题，而是更接近专家在真实工作中承担的完整专业工作单元',
 '这些环境覆盖了更广泛的生产级工作流，任务设计也更加贴近工程与研究工作的实际开展方式，其中部分任务即使由经验丰富的工程师完成，也需要数天时间',
 '例如，在一项机器学习基础设施任务中，模型可能会获得与工程师相同的工作环境，包括计算集群、存储系统、内部文档、代码库和实验结果等资源',
 '模型需要定位训练技术栈中的性能瓶颈，实施优化方案，运行实验，并在保证结果正确性的前提下，实现可量化的端到端加速',
 '通过在此类高复杂度环境中训练，推动模型逐步具备端到端负责并完成复杂工作的能力，而不再依赖用户拆解问题并逐步监督执行',
 '随着智能体能力提升，后训练规模化的主要难点也逐渐从模型本身转移到任务环境',
 '一个真正有价值的任务环境必须可执行、可验证，并贴近真实的专业工作；同时，我们需要的不是少量人工构建的环境，而是大规模、多样化的任务环境',
 '为实现这一过程的规模化，我们构建了能够端到端合成任务环境的流水线，并针对部分任务进一步自动生成强化学习所需的奖励信号',
 '研究智能体从真实工作中收集任务模式，并将其转化为可运行的长程任务环境，其中包含多步骤依赖关系和隐藏状态；随后，评审智能体会实际尝试完成每项任务，以验证任务是否确实可解',
 '验证器在不访问参考解法的前提下生成；与此同时，系统会利用求解轨迹识别并消除奖励捷径',
 '只有同时通过 Oracle、空操作和未解决状态检查的验证器，其生成的二值奖励信号才足够可靠，可直接用于模型训练',
 'GLM-5.3 延续了 GLM-5.2 中引入的强化学习策略，包括结合上下文压缩机制的 SAO，使能力增益能够在长程任务中持续体现，而非仅局限于短程任务',
 '这些提升同时反映在编程任务和通用智能体任务中：GLM-5.3 在 Terminal-Bench 3.0 上的得分由 4.6 提升至 28.3，在 DeepSWE v1.1 上由 46.2 提升至 66.9，在 Agents’ Last Exam 上由 23.8 提升至 28.5',
 '目前，这些流水线仍需要较多人工参与',
 '进一步提升任务环境生成与验证过程的自主化程度，是我们下一阶段的重点方向之

#### 2. Embedding the Text Chunks

In [6]:
import requests
import numpy as np

def embed_text(docs, model="embedding-3", embed_dim=2048):
    """
    将 docs 使用 embedding-3 嵌入之后转成 embedding 向量
    """

    url = "https://open.bigmodel.cn/api/paas/v4/embeddings"

    payload = {
        "model": model,
        "input": docs,
        "dimensions": embed_dim
    }
    headers = {
        "Authorization": f"Bearer {ZHIPU_APIKEY}",
        "Content-Type": "application/json"
    }

    response = requests.post(url, json=payload, headers=headers)

    print("嵌入模型成功，结果为：")
    print(response.text)

    # 获取 embedding 向量
    result = response.json()        # 将其变成字典
    embeddings = [
        item["embedding"]
        for item in sorted(
            result["data"],
            key=lambda x: x["index"]
        )
    ]

    embeds = np.array(embeddings)

    return embeds

将我们的文档嵌入

In [7]:
embeds = embed_text(texts)
print("其形状为：", embeds.shape)

嵌入模型成功，结果为：
{"data":[{"embedding":[-0.011767769,0.005402339,-0.014576784,-0.022191223,-0.025521914,-0.0073435693,0.000051885276,0.003952686,-0.008010711,0.016803933,0.024719337,-0.0030522963,0.016513,-0.006119641,0.026344553,-0.011777801,-0.006962346,-0.01273086,-0.021428777,0.019502593,0.01971327,0.0047552623,-0.006556042,0.032143164,0.0118781235,-0.058788683,0.0031576345,-0.032544453,0.0077749537,0.009500492,-0.0035238098,0.08126081,-0.0053772586,0.008366854,-0.0036065753,0.032283615,-0.03362793,-0.007514117,0.035835013,-0.00023105407,0.004090629,0.0028115236,0.0067416374,-0.029273955,0.03140078,-0.03441044,-0.032283615,0.008326725,0.02660539,-0.025521914,0.013503339,-0.01779712,0.008437079,-0.0031977633,-0.007790002,0.007529165,0.0022321641,0.0064356555,0.010177665,0.028090157,-0.034891985,-0.0029695309,0.016412677,0.02381644,0.011687512,-0.03244413,-0.008356822,-0.024619015,-0.038022034,0.012179089,0.024217727,0.09454344,0.01597126,0.0012483817,-0.008261516,0.015078395,-0.007198102

#### 3. Building The Search Index

在搜索之前，我们需要创建搜索索引来加快检索的速度。向量的索引与传统数据库的索引不同，传统数据库索引实现的常见技术是B+树和 Hash，而向量的索引使用的是 Flat、IVF 和 HNSW 等技术。其中：

- **Flat(暴力搜索)**：遍历所有向量，逐一计算距离
- **IVF（Inverted File Index, 倒排文件索引）**：先对向量聚类，查询时只搜索部分相关的聚类
- **HNSW（Hierarchical Navigable Small World, 分层导航小世界图）**: 将向量组织成多层图，通过图上的邻居关系逐步接近目标向量


我们这里使用 FAISS 来加快搜索, 它负责存储、索引和搜索向量，并且它实现了上面三种技术。它有以下特点：

- 支持高维向量相似度搜索
- 支持欧氏距离、内积等度量方式
- 提供精确最近邻搜索和近似最近邻搜索
- 底层主要使用 C++ 实现，提供 Python 接口
- 部分索引和搜索算法支持 GPU 加速

我们来说一下 FAISS 的索引类型，如下表所示

| 索引类型            | 说明                                                     | 是否需要训练 |
|-----------------|--------------------------------------------------------|---|
| `IndexFlatL2`   | 基于欧氏距离（L2）的精确最近邻搜索，返回距离最近的 K 个向量                       | 不需要 |
| `IndexFlatIP`   | 基于内积（Inner Product）的精确最近邻搜索，返回内积最大的 K 个向量              | 不需要 |
| `IndexIVFFlat`  | 基于倒排文件的近似最近邻搜索，通过 K-Means 将向量划分到多个聚类中，搜索时只访问部分聚类，减少计算量 | 需要 |
| `IndexIVFPQ`    | 结合倒排文件和乘积量化（PQ）的近似最近邻搜索。先通过聚类缩小搜索范围，再利用 PQ 对向量进行压缩     | 需要 |
| `IndexHNSWFlat` | 基于 HNSW 的近似最近邻搜索，通过构建多层邻居图，在图中逐步寻找与查询向量最接近的节点。         | 不需要 |

⚠️⚠️⚠️**注意，这里的训练不是训练 Embedding 模型，而是说** <span style="color: red">**有些 FAISS 索引需要通过训练来学习向量的聚类中心或量化参数，从而建立更高效的搜索结构**</span>


下面我们为嵌入到 embedding-3 模型的向量创建搜索索引，这里使用 `IndexFlatL2` 来创建

In [8]:
import faiss

dim = embeds.shape[1]

# 创建指定维度的索引
index = faiss.IndexFlatL2(dim)

# 将向量添加到索引中
index.add(np.float32(embeds))

让我们看看 `IndexFlatL2` 是否可训练？

In [9]:
print(index.is_trained)

True


#### 4. Search the Index

我们现在可以用我们想去 query 的东西搜索数据集了，让我们先来定义一个 search 函数

In [15]:
import pandas as pd

def search(query, number_of_results=3):

    # 1. 获取查询向量的嵌入
    query_embed = embed_text(query)

    # 2. 转换成二维 float32 数组
    query_embed = np.asarray(
        query_embed, dtype=np.float32
    )

    # 3. 检索最近邻的向量: 找3个
    # index.total: 表示 FAISS 索引中实际存储的向量数
    k = min(number_of_results, index.ntotal)

    distances, similar_item_ids = index.search(query_embed, k)

    # 3. 格式化结果
    texts_np = np.array(texts)
    results = pd.DataFrame(data={'texts': texts_np[similar_item_ids[0]],
                                 'distance': distances[0]})

    return results

In [16]:
query = "GLM 5.3 相比 GLM 5.2 强在什么地方？"
results = search(query)

嵌入模型成功，结果为：
{"data":[{"embedding":[-0.04693991,0.025635732,-0.015016793,0.024398146,-0.016663227,0.00052866794,0.00016609358,0.019370448,-0.017856615,0.012033325,0.02300586,-0.015315141,0.011392431,-0.0016878699,0.020055542,-0.028133007,-0.014652148,-0.015469839,-0.026121927,-0.01539249,0.028928598,0.008116141,0.0011554036,-0.0020483725,0.030541882,-0.033392753,0.010685239,0.0048094625,-0.01724887,-0.01059684,0.016663227,0.06957007,-0.0027044592,0.014022304,0.010701814,0.034232542,-0.018464359,0.0010711482,0.029613692,-0.011602379,0.02780151,0.024110848,-0.002853633,-0.011955976,0.008613385,-0.028464504,-0.02026549,-0.0016892512,0.04015528,-0.01829861,-0.0020387038,-0.033989444,0.027735211,-0.0060940115,-0.011337182,0.017812414,-0.019005802,-0.012607919,0.012232223,0.0023494817,-0.0100995945,-0.03076288,0.013403511,0.03337065,0.0059945625,0.0010069208,-0.026630223,-0.0031271174,-0.010989111,-0.023492055,-0.0023260007,0.08309513,0.012729468,-0.007983542,-0.0041519944,0.0056188665,-0.018

In [17]:
results

,texts,distance
0,GLM-5.3 延续了 GLM-5.2 中引入的强化学习策略，包括结合上下文压缩机制的 SA...,0.779668
1,在 GLM-5.3 的训练中，我们进一步推进了任务环境的规模化建设，使训练任务不再只是传统的...,0.895906
2,这些提升同时反映在编程任务和通用智能体任务中：GLM-5.3 在 Terminal-Benc...,1.104497


我们可以比较传统的关键字搜索和上述的语义搜索，在这里，我们使用 BM25 算法，它是一种词汇搜索的算法.

这里，我们为了更好的对中文进行分词，使用 `jieba` 库

In [22]:
!pip install jieba

In [23]:
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
import jieba

def bm25_tokenizer(text):
    return [
        token.strip()
        for token in jieba.lcut(text.lower())
        if token.strip()
    ]

tokenized_corpus = []
for passage in tqdm(texts):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM25Okapi(tokenized_corpus)

/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
/usr/local/lib/python3.13/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


  0%|          | 0/19 [00:00<?, ?it/s]

Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
DEBUG:jieba:Dumping model to file cache /tmp/jieba.cache
Loading model cost 0.629 seconds.
DEBUG:jieba:Loading model cost 0.629 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.


定义关键字检索函数

In [24]:
def keyword_search(query, top_k=3, num_candidates=15):
    print("输入的问题为：", query)

    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print("Top 3 BM25 Hits:")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

In [25]:
keyword_search(query)

输入的问题为： GLM 5.3 相比 GLM 5.2 强在什么地方？
Top 3 BM25 Hits:
	8.465	GLM-5.3 延续了 GLM-5.2 中引入的强化学习策略，包括结合上下文压缩机制的 SAO，使能力增益能够在长程任务中持续体现，而非仅局限于短程任务
	4.869	在 GLM-5.3 的训练中，我们进一步推进了任务环境的规模化建设，使训练任务不再只是传统的编程题，而是更接近专家在真实工作中承担的完整专业工作单元
	4.535	这些提升同时反映在编程任务和通用智能体任务中：GLM-5.3 在 Terminal-Bench 3.0 上的得分由 4.6 提升至 28.3，在 DeepSWE v1.1 上由 46.2 提升至 66.9，在 Agents’ Last Exam 上由 23.8 提升至 28.5


可以看到，查询出来的第一个结果仅仅是原文的内容，不是在回答我的问题

### 1.1.2 Dense 检索的局限性（Caveats of dense retrieval）

我们需要了解 Dense 检索的缺点并且要知道怎么处理它们。如果搜索出来的文本没有包含答案会发生什么呢？举个例子来说

In [26]:
query = "地球的质量是多少？"
results = search(query)

嵌入模型成功，结果为：
{"data":[{"embedding":[0.02695645,-0.027266057,-0.000010043079,-0.0005382647,-0.012384281,-0.04978481,0.017492795,0.011538021,-0.025078168,-0.024376392,0.027596304,0.026853248,-0.011197453,0.007714375,0.01808105,-0.016976785,-0.0011171653,-0.0050027333,0.003268934,0.00441706,-0.014623771,0.0061973003,0.0012403631,-0.012590685,0.0016512374,-0.02602763,0.014902418,0.012219157,-0.033375636,-0.021424806,0.020805592,0.08152985,0.0067803934,-0.007239644,0.007822737,-0.009040525,0.008060102,-0.02602763,0.0061456994,-0.018710583,0.012033393,0.0167807,0.0050698146,-0.033375636,0.021125518,-0.007549251,-0.04284961,0.0068371547,0.024809841,-0.024087425,-0.023963584,-0.029433306,-0.012188196,-0.021672491,0.010072548,0.008607075,0.014190322,0.016491733,0.027018372,0.017368954,-0.007977541,-0.04767948,-0.0076111723,0.03349948,-0.021321602,0.012456522,-0.022209143,-0.021878896,0.014984979,-0.017121268,0.01808105,0.036801953,0.028463205,0.024995606,0.048752785,0.0046621654,-0.020537265,0.0

In [27]:
results

,texts,distance
0,目前，这些流水线仍需要较多人工参与,1.210638
1,该基准覆盖多种任务类别，并将智能体置于复杂的本地开发环境中,1.296233
2,进一步提升任务环境生成与验证过程的自主化程度，是我们下一阶段的重点方向之一,1.315736


`distance` 表示两个 query 和 answer 向量之间的距离，因为文本中从来没有介绍“地球的质量”这个内容，所以我们可以看出查询出的内容与 query 没有任何关联。

Dense 检索的另一个缺点：用户如果想了解这个问题会精确匹配哪个段落，该怎么办？这就是关键字检索的好处了。解决此问题的方法是我们要使用混合检索（Hybrid Search）,它包含语义检索和关键词检索。

Dense 检索有一个域的限制，举个例子来说，如果你在维基百科的数据上训练检索模型，那么将其部署到法律领域上，这个模型表现得不会很好。

最后一个要说明的是，如果检索的答案分布在多条信息中，我们该怎么办呢。这就要考虑到 Dense 检索系统设计的重要性了：长文本切块的最好方式是什么？为什么我们要先进行切块？

### 1.1.3 改善 Dense 检索

那么，我们怎么改善 Dense 检索的质量呢？下面介绍两个方法

- 长文本切块策略
- 微调 Embedding 模型

#### 1.1.3.1 长文本切块策略

由于大模型上下文长度的限制，我们不能将全部的文本一次喂给模型，所以我们要对长文本进行切块，下面是切块的策略

| 切块策略 | 英文名称 | 原理 | 优点 | 缺点 |
|---|---|---|---|---|
| 固定长度切块 | Fixed-size Chunking | 按固定字符数或 Token 数切分 | 简单、高效 | 容易破坏语义 |
| 滑动窗口切块 | Sliding Window Chunking | 固定长度切块，相邻 Chunk 保留重叠内容 | 减少上下文丢失 | 存在文本冗余 |
| 递归切块 | Recursive Chunking | 按段落、换行、句子等分隔符逐级切分 | 尽可能保持文本结构 | 无法完全保证语义完整 |
| 句子切块 | Sentence-based Chunking | 按句子边界切分，并组合相邻句子 | 句子完整 | Chunk 长度不均匀 |
| 语义切块 | Semantic Chunking | 根据 Embedding 的语义相似度确定切分位置 | 语义连续性好 | 计算成本较高 |
| 文档结构切块 | Document-based Chunking | 根据标题、章节、表格、代码块等结构切分 | 保留文档结构 | 依赖文档格式 |
| 层次化切块 | Hierarchical Chunking | 将文档组织为父 Chunk 和子 Chunk | 兼顾检索精度与上下文 | 实现较复杂 |
| LLM 智能切块 | LLM-based Chunking | 利用 LLM 判断主题及语义边界 | 适合复杂文档 | 成本高、速度慢 |

#### 1.1.3.2 微调嵌入模型

Dense 检索优化的更多的是要优化文本的嵌入，而不仅仅是token的嵌入。

我们来看这样一个例子，句子 “Interstellar（星际穿越）于2014年10月26日在洛杉矶首映.”。与此句子相关的查询是：

- Interstellar 的上映日期？
- Interstellar 什么时候首映？

微调过程就是让这些查询的嵌入向量更接近与目标句。它也需要看到对于查询的负样本（它与句子毫不相关），负样本为：Interstellar 的演员阵容。

我们现在有三个样本对了，两个正样本和一个负样本。让我们假设，这三个查询都与原来的文档有相同的距离，之后我们将进行**微调**：*让正样本离的更近，让负样本离得更远*。如下图所示

<center>
<img src="./resources/fine_tuning.png">
</center>

## 1.2 重排名（Reranking）

为了改善检索质量，我们将搜索结果进行重排名，如下图所示，重排序在第二阶段

<center>
<img src="./resources/reranking_two_stage.png">
</center>

实际情况下，上图所示的这个流水线极大地改善了搜索的质量。像 MIRACL 这样的多语言基准测试中，Reranker 可以将模型的表现从 36.5 提升至 62.8，它的测量指标为 NDCG@10

接下来，我们看一下 Reranker 模型是怎么工作的？

构建 LLM 搜索的 rerankers 最常见的方式 **Cross-Encoder**，它的核心思想是：**将用户查询和候选文档拼接起来，一起输入 BERT，让模型计算两者之间的相关性。** 这种将搜索表示为相关性评分的方式归根结底是一个分类问题。给定输入，模型会输出一个 0-1 的得分，其中 0 表示不相关，1 表示强相关。

In [28]:
import requests

url = "https://open.bigmodel.cn/api/paas/v4/rerank"

payload = {
    "model": "rerank",
    "query": "GLM 5.3 相比 GLM 5.2 强在什么地方？",
    "documents": [
        "GLM-5.3 延续了 GLM-5.2 中引入的强化学习策略，包括结合上下文压缩机制的 SAO，使能力增益能够在长程任务中持续体现，而非仅局限于短程任务",
        "在 GLM-5.3 的训练中，我们进一步推进了任务环境的规模化建设，使训练任务不再只是传统的编程题，而是更接近专家在真实工作中承担的完整专业工作单元",
        "这些提升同时反映在编程任务和通用智能体任务中：GLM-5.3 在 Terminal-Bench 3.0 上的得分由 4.6 提升至 28.3，在 DeepSWE v1.1 上由 46.2 提升至 66.9，在 Agents’ Last Exam 上由 23.8 提升至 28.5"],
    "top_n": 4
}
headers = {
    "Authorization": f"Bearer {ZHIPU_APIKEY}",
    "Content-Type": "application/json"
}

response = requests.post(url, json=payload, headers=headers)

print(response.text)

{"created":1792723632,"id":"2026102310471295e3eee81ccc4480","request_id":"2026102310471295e3eee81ccc4480","results":[{"index":2,"relevance_score":1.0},{"index":1,"relevance_score":1.0},{"index":0,"relevance_score":1.0}],"usage":{"completion_tokens":0,"prompt_tokens":353,"total_tokens":353}}


In [31]:
resp_json = response.json()

results = resp_json["results"]
for item in results:
  print(f"index: {item["index"]}")
  print(f"relevance_score: {item["relevance_score"]:.5f}")
  print()

index: 2
relevance_score: 1.00000

index: 1
relevance_score: 1.00000

index: 0
relevance_score: 1.00000



可以看到经过重排之后相关性分数都为1（强相关），这里因为所提供的示例文本内容比较短，且搜索出的内容确实与 query 强相关，这里需要注意一点的是，它已经进行了重排，因为文档对应的 index 已经改变了

## 1.3 检索的评估指标

评估语义检索的指标一般在 IR（Information Retrieval: 信息检索）领域中。下面让我们介绍以下评估指标：

- Precision@K：精确率
- Recall@K：召回率
- MRR：平均倒数排名
- AP（Average Precision）: 平均精确率
- MAP（Mean Average Precision）：平均精确率均值
- NDCG(Normalized Discounted Cumulative Gain): 归一化折损累积增益

你可能想知道为什么相同的操作都叫 “mean” 和 “average”，这似乎是一个**美学的选择**，因为 MAP 总比 average average precision 好得多。另外，*<span style="color: red">NDCG 比 AP 和 MAP 指标更微妙</span>*，**它评估的不是检索结果中那些二分类（相关和不相关）的文档，而是那种多级相关性（例如 高度相关，部分相关，不相关等等）的文档**

### 1.3.1 Precision@K

`Precision@K`：它**指的是在你的搜索结果中前 K 个结果中有多少个比例是相关的文档**，其公式为：

$$
Precision@K = \frac{前 K 个结果中的相关文档数量}{K}
$$

---

对于如下的例子来说：

| 排名	| 文档	| 是否相关 |
|---|---|---|
| 1	| 文档 A	| 相关	|
| 2	| 文档 B	| 不相关	|
| 3	| 文档 C	| 相关	|
| 4	| 文档 D	| 不相关	|
| 5	| 文档 E	| 相关	|

那么，

$$
Precision@1 = \frac{1}{1} = 1
$$

$$
Precision@3 = \frac{2}{3} = 0.67
$$
其中，2 代表前 3 个之中相关文档的数量


$$
Precision@5 = \frac{3}{5} = 0.6
$$
其中，3 代表前 5 个之中相关文档的数量

### 1.3.2 Recall@K

**Recall@K** 表示在检索结果的前 K 个文档中，成功召回相关文档占所有相关文档的比例，计算公式为：

$$
Recall@K = \frac{前 K 个结果中的相关文档数量}{所有相关文档的总数量}
$$

---


计算示例

假设用户输入一个 Query：

> RAG 有哪些常见的检索优化技术？

知识库中共有 **10 个相关文档**，检索模型返回 Top 5 个文档：

| 排名 | 文档 | 是否相关 |
|:---|:---|:---|
| 1 | Hybrid Search | 相关 |
| 2 | Query Expansion | 相关 |
| 3 | Python 基础语法 | 不相关 |
| 4 | Reranking | 相关 |
| 5 | Query Rewriting | 相关 |

**计算 Recall@5：**

$$
Recall@5 = \frac{4}{10} = 0.4
$$


这意味着模型在前 5 个检索结果中，成功召回了全部相关文档的 **40%**。

### 1.3.3 MRR(Mean Reciprocal Rank)

先来说 **RR（Reciprocal Rank）倒数排名**，对于单个 Query 来说：

$$
RR  = \frac{1}{rank}
$$

其中，** $rank$ 表示第一个相关文档的排名**，若检索中无相关文档，则 $RR = 0$。

再说 **MRR（Mean Reciprocal Rank, 平均倒数排名）**，它是对于多个 Query 来说的，其公式为

$$
MRR = \frac{1}{Q} \sum_{i=1}^Q \frac{1}{rank_i}
$$

其中：

- $Q$ 表示 Query 的总数量
- $rank_i$：表示第 $i$ 个Query 的第一个相关文档的排名

如果某个 Query 未找到相关文档，该 Query 的 RR 记为 0

---
下面我们来看一个例子

假设有三个 Query：

| Query | 第一个相关文档的位置 | RR |
|:---|:---|---:|
| Python 的作者是谁？ | 第 1 名 | 1.000 |
| Transformer 是谁提出的？ | 第 2 名 | 0.500 |
| BERT 是哪一年发布的？ | 第 4 名 | 0.250 |

**计算 MRR：**

$$
MRR = \frac{1+\frac{1}{2}+\frac{1}{4}}{3} \approx 0.5833
$$

这意味着模型在这三个 Query 上的**平均倒数排名为 0.5833**。


### 1.3.4 AP(Average Precision)

**AP 的计算规则是：只取相关文档所在位置的 Precision@K，然后求其平均值**

AP 的常用定义为：
$$
AP(q) = \frac{1}{R_q}\sum_{k=1}^N Precision@k \cdot rel(k)
$$

其中：

- $R_q$：该 Query 的相关文档总数
- $N$: 文档的总数
- $rel(k)$：表示第 $k$ 个文档是否相关，相关为 1，否则为 0

---

我们还拿 1.3.1 节的例子来说，这里我又在表格最后加了一列 `Precision@K`, 如下表所示

| 排名	| 文档	| 是否相关  | Precision@K  |
|---|---|-------|--------------|
| 1	| 文档 A	| ✅ 相关	 | $1/1 = 1.00$ |
| 2	| 文档 B	| 不相关	  | $1/2 = 0.50$ |
| 3	| 文档 C	| ✅ 相关	 | $2/3 = 0.67$ |
| 4	| 文档 D	| 不相关	  | $2/4 = 0.50$ |
| 5	| 文档 E	| ✅ 相关	 | $3/5 = 0.60$ |

接下来，我们计算AP

$$
AP = \large\frac{\frac{1}{1} + \frac{2}{3} + \frac{3}{5}}{3} \approx 0.7556
$$


让我们来换一种（理想化）排列方式：

| 排名	| 文档	| 是否相关  | Precision@K  |
|---|---|-------|--------------|
| 1	| 文档 A	| ✅ 相关	 | $1/1 = 1.00$ |
| 2	| 文档 B	| ✅ 相关	  | $2/2 = 1.00$ |
| 3	| 文档 C	| ✅ 相关	 | $3/3 = 1.00$ |
| 4	| 文档 D	| 不相关	  | $3/4 = 0.75$  |
| 5	| 文档 E	| 不相关	 | $3/5 = 0.60$ |


接下来，我们再计算AP

$$
AP = \large\frac{\frac{1}{1} + \frac{2}{2} + \frac{3}{3}}{3} = 1
$$


可以看到，AP 越接近 1, 排序的质量越好

### 1.3.5 MAP(Mean Average Precision)

上面介绍的 AP 衡量的是**一个 Query 的排序质量**，而 MAP 衡量的是**多个 Query 的平均排序质量**。

那么，在 Reranker 的评估中，**MAP 最适合用于那些一个 Query 对应多个相关文档，并且希望所有相关文档尽可能排在前面的检索任务**

**那什么类型的 Query 适合使用 MAP呢？**

Query 类型可以是多文档检索、综合知识查询、技术资料查询和多级相关性查询等等，唯一事实和精确定位的查询不适合 MAP。其类型对应的示例及是否适合 MAP 的原因如下表所示：

| Query 类型 | 示例                       | 是否推荐 MAP | 原因                       |
|----------|--------------------------|----------|--------------------------|
| 多文档检索    | Transformer 有哪些常见的注意力机制？ | 👍🏻推荐   | 多篇文档都可能相关                |
| 综合知识查询   | RAG 有哪些常见的优化方法？          | 👍🏻推荐   | 需要检索多个知识片段               |
| 技术资料查询   | Python 有哪些常见的装饰器？        | 👍🏻推荐   | 可能有多个相关文档                |
| 唯一事实查询   | Python 的作者是谁？            | - 不优先    | 如果只需找到一个正确答案，*MRR* 更直观   |
| 精确定位查询   | 订单 123456 的状态是什么？        | - 不优先    | 通常只需找到一条准确记录即可           |
| 多级相关性查询  | 哪些文档最适合学习LangGraph？      | ✅ 可使用    | 如果有高度相关、部分相关等层级，NDCG 更适合 |

表中提到了 *MRR*，它是平均倒数排名，全称为 Mean Reciprocal Rank，它用于衡量**第一个相关文档在检索结果中的排名位置**，公式如下：

$$
MRR = \frac{1}{Q} \sum_{i=1}{Q} \frac{1}{rank_i}
$$

其中：
- $Q$：Query 总数量。
- $rank_i$：第 i 个 Query 的第一个相关文档排名。
- 如果某个 Query 未找到相关文档，该 Query 的 RR 记为 0。

接下来，我们讲 MAP 的公式定义以及示例

MAP 基于 AP 计算，其公式为：

$$
MAP = \frac{1}{Q} \sum_{q=1}^Q AP(q)
$$

其中：

- $Q$ 表示查询的总数量
- $AP(q)$ 表示第 $q$ 个查询的 AP 值


假设我们测试了 3 个 Query:

| Query       | AP|
|-------------|---|
| 星际穿越什么时候首映？ |	0.7556 |
 | 星际穿越的导演是谁？  |	0.9000 |
 | 星际穿越的主演有哪些？ |	0.8000 |

那么

$$
MAP = \frac{0.7556 + 0.9 + 0.8}{3} \approx 0.8185
$$

### 1.3.6 NDCG(Normalization Discount Cumulative Gain)

NDCG 用于衡量模型的排序结构与理想排序之间的接近程度。与前面介绍的 MAP 不同，**NDCG 不仅关注文档是否相关，还考虑了文档的相关程度以及相关文档出现的位置。**

NDCG 的三个计算步骤分为：`DCG` -> `IDCG` -> `NDCG = DCG/IDCG`

1. 计算 DCG（Discount Cumulative Gain）

DCG 的常见公式为

$$
DCG@K = \sum_{i=1}^K \frac{2^{rel_i} - 1}{\log_2 (i + 1)}
$$

其中：

- $K$ 表示评估前 K 个排序结果
- $i$ 表示当前文档的排名，从 1 开始
- $rel_i$ 表示第 $i$ 个文档的真实相关性等级
- $2^{rel_i} - 1$ 表示文档带来的收益


例如，我们这样表示收益

| 相关性等级 | 描述	  | $Gain = 2^{rel} − 1$ |
|-------|------|----------------------|
| 0     | 不相关  | 0                    |
 | 1     | 部分相关 | 1                    |
 | 2     | 比较相关 | 3                    |
 | 3     | 高度相关 | 7                    |


那**为什么要除以 $\log_2(i+1)$ 呢？因为越靠后的文档，被用户看到或使用的可能性通常越低。**

例如，同样是高度相关的文档（$rel = 3$），不同的排名带来的收益如下表：

| 排名 | 计算              | 折损后的收益      |
|----|-----------------|-------------|
| 1  | $7 / \log₂(2)$	 | 7.000 |
| 2  | $7 / \log₂(3)$	 | 4.417 |
| 3  | $7 / \log₂(4)$	 | 3.500 |
| 4  | $7 / \log₂(5)$	 | 3.015 |
| 5  | $7 / \log₂(6)$  | 2.708 |

2. 计算 IDCG(Ideal Discount Cumulative Gain)

IDCG 表示**理想情况下能够获得的最大的 DCG**。

计算方法很简单：按照真实的相关性等级，从高到低对所有候选文档排序，再计算 DCG@K。公式如下：

$$
IDCG@K = \sum_{i=1}^K \frac{2^{rel_i^*} - 1}{\log_2(i + 1)}
$$

其中，$rel^*$ 表示按**真实相关性降序排列后的等级**

3. 计算 NDCG(Normalized Discount Cumulative Gain)

公式如下：

$$
NDCG@K = \frac{DCG@K}{IDCG@K}
$$

则：

- $NDCG = 1$：达到理想排序
- $NDCG 越接近 1$：排序质量越高
- $NDCG 越接近 0$：排序质量越差

这里我们假设：相关性标签为非负数，且 $IDCG > 0$。若全部文档均不相关，通常约定 NDCG 为 0.

这里我们展示一个计算示例，仅拿一个 Query 来举例，多个 Query 将其进行求和即可

假设用户 Query 是 “《星际穿越》什么时候首映？”，那经过 Reranker 重排序后，得到如下的结果，⚠️⚠️⚠️**注意，这里的相关性等级是由人工标注提供的，并不是模型自己的预测分数**

| 排名 | 文档 | 真实相关性 $rel$ | $Gain = 2^{rel} - 1$ |
| ---|---|-------------|----------------------|
| 1	| 文档 A | 1           | 1                    |
| 2	| 文档 B | 0           | 0                    |
| 3	| 文档 C | 3           | 7                    |
| 4	| 文档 D | 2           | 3                    |
| 5	| 文档 E | 0           | 0                    |


我们先来计算 $DCG@5$

$$
DCG@5 = \frac{1}{\log_2(2)} + \frac{0}{\log_2(3)} + \frac{7}{\log_2(4)} + + \frac{3}{\log_2(5)} + + \frac{0}{\log_2(6)} \approx 5.7920
$$

理想的相关性排序应该是 32100, 则对应的增益为 73100，则

$$
IDCG@5 = \frac{7}{\log_2(2)} + \frac{3}{\log_2(3)} + \frac{1}{\log_2(4)} + + \frac{0}{\log_2(5)} + + \frac{0}{\log_2(6)} \approx 9.3928
$$

最终, 计算 $NDCG@5$

$$
NDCG@5 = \frac{DCG@5}{IDCG@5} = \frac{5.7920}{9.3928} \approx 0.6166
$$

也就是说，Reranker 模型当前排序获得了约 61.66% 的理想折损累积增益

我们来总结一下， Reranker 应该使用哪种指标来衡量呢？

| 评估指标 | 核心关注点 | 适用场景 | 推荐程度 |
|:---|:---|:---|:---|
| **NDCG@K** | **高相关性文档是否排在更前面** | 多级相关性排序、RAG | **优先推荐** |
| **MRR@K** | **第一个相关文档出现得有多早** | FAQ、事实问答、单答案检索 | **推荐** |
| **MAP@K** | **所有相关文档的整体排序质量** | 一个 Query 对应多个相关文档 | **推荐** |
| **Recall@K** | **前 K 个结果覆盖多少相关文档** | RAG、多文档检索 | **推荐** |
| **Precision@K** | 前 K 个结果有多少是相关文档 | 关注结果纯度的检索 | 可选 |

# 二、RAG（Retrieval-Augmented-Generation）

LLM 的快速普及很快使人们可以询问它们问题并且期望它给出实际的答案。虽然模型可以正确的回答某些问题，但还是避免不了他们自信的输出不正确的回答。为了弥补这种现象，业界通常采用的方式是 RAG，如下图所示

<center>
<img src="./resources/rag_pipeline.png">
</center>

## 2.1 从语义搜索到 RAG

让我们把我们的搜索系统转换成 RAG 系统，我们可以在 Search 流水线的末尾添加 LLM 模型，我们将问题和检索出的文档再次送给 LLM 模型，让其通过搜索出的上下文来回答问题。下图展示了这一过程：

<center>
<img src="./resources/search_to_rag.png">
</center>

下面，我们使用本地模型来阐述 RAG，分为以下三步

1. 加载生成模型
2. 加载嵌入模型
3. 使用 RAG 提示词调用 LLM

1. 加载生成模型

我们依旧使用 `Phi-3-mini-4k`，我们先来下载它

In [32]:
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

--2026-10-23 02:53:53--  https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
Resolving huggingface.co (huggingface.co)... 13.35.202.40, 13.35.202.34, 13.35.202.121, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.40|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/662698108f7573e6a6478546/a9cdcf6e9514941ea9e596583b3d3c44dd99359fb7dd57f322bb84a0adc12ad4?user_id=public&X-Xet-Cas-Uid=public&xip=9RSk3GzB_FA&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Phi-3-mini-4k-instruct-fp16.gguf%3B+filename%3D%22Phi-3-mini-4k-instruct-fp16.gguf%22%3B&Expires=1792727633&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjYyNjk4MTA4Zjc1NzNlNmE2NDc4NTQ2L2E5Y2RjZjZlOTUxNDk0MWVhOWU1OTY1ODNiM2QzYzQ0ZGQ5OTM1OWZiN2RkNTdmMzIyYmI4NGEwYWRjMTJhZDRcXD91c2VyX2lkPXB1YmxpYyZYLVhldC1DYXMtVWlkPXB1YmxpYyZ4aXA9OVJTazNHekJfRkEm

之后我们使用 `llama.cpp`，`llama-cpp-python` 和 LangChain 来加载此模型

In [34]:
from langchain_community.llms import LlamaCpp

llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

/tmp/ipykernel_2705/2810872154.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import LlamaCpp


2. 加载嵌入模型

现在让我们加载嵌入模型，我们使用 `BAAI/bge-small-en-v1.5 model`

In [36]:
%pip install -U langchain-huggingface

In [37]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name='thenlper/gte-small'
)

modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/68.1k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 66.7MB            

...

tokenizer_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

我们现在可以使用嵌入模型来设置我们的向量数据库了

In [38]:
from langchain_community.vectorstores import FAISS

# 创建本地的向量数据库
db = FAISS.from_texts(
    texts=texts,
    embedding=embedding_model
    )

3. 使用 RAG 提示词调用 LLM

提示词模板在 RAG 中很重要，我们将搜索出来的相关文档放到一个名叫 `context` 的变量中，那么它可以改善 LLM 的检索质量。

提示词模板如下

In [41]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. 创建提示词模板
template = """<|user|>
Relevant information:
{context}

Provide a concise answer the following question using the relevant information provided above:
{question}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

# 2. 创建检索器
retriever = db.as_retriever(
    search_kwargs={"k": 3}
)

# 3. 格式化检索到的文档

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. 构建 RAG 流水线
rag = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

现在，我们来调用 rag

In [42]:
rag.invoke('GLM 5.3 相比 GLM 5.2 强在哪里？')

' GLM 5.3相比 GLM 5.2在结合上下文压缩机制的 SAO 方面强化了学习能力，在长期任务中表现更好。同时，通过规模化建设，使得任务环境更接近真实工作，并在编程任务和通用智能体任务上取得了显著提升的成绩，比如在 Terminal-Bench 3.0 上提升了28.3，在 DeepSWE v1.1 上提升了66.9，以及在 Agents’ Last Exam 上提升了28.5。'

可以看到，我们经过RAG之后，比Dense 和 BM25 检索出来的结果更像是 LLM 再回答我们的问题。另外，需要注意的是，**<span style="color: red">这段代码执行了 3 分钟！！！这是比较可怕的！！!</span> **因为我们的文本和问题体量还特别小，所以我们在后续的章节中，我们会介绍推理借宿的部分

## 2.2 高级的 RAG 技术

我们来介绍几种技术来提升 RAG 系统的性能

### 2.2.1 查询重写（Query rewriting）

如果 RAG 系统是一个聊天机器人，那么你问的问题越详细，或者说它参考的上下文越长，上述所介绍的简单的 RAG 实现的效果可能越差劲。因此，我们可以使用 LLM 来重写查询，为的是逐步的获取正确的信息或者回答。

让我们来看一个例子：

```text
用户提问：“我们明天要交一篇作文，题目是写某种动物。我挺喜欢企鹅的，可以写企鹅；不过写海豚也行。海豚算动物吗？大概算吧。那就写海豚吧。比如，它们住在哪里呢？”
```

我们可以改写成：

```text
海豚住在哪里？
```

查询重写可以通过提示词或者调用对应的 API 来完成

### 2.2.2 多查询 RAG

我们可以对一个问题进行多步查询，举个例子：

```text
用户提问：“比较英伟达 2020 和 2023 年的财报”
```

对于这个问题，我们也许会找到一篇文档包含这两年的财报，但是，我们最好将此查询分开成以下几个问题：

```text
Query1: "英伟达 2020 年财报"
Query2: "英伟达 2023 年财报"
```

### 2.2.3 多步骤 RAG（Multi-hop Query）

一个更高级的问题也许需要一系列的查询。我们来看这个例子：

```text
用户提问：“2023 年最大的汽车制造业都有哪些？它们都会生产电动汽车（EV，Electric Vehicles）么？”
```

为了回答这个问题，这个系统首先会搜索：

```text
Step 1 -> Query 1: "2023 年最大的汽车制造业都有哪些？"
```

随后，基于此问题的结果（这个结果是 Toyota, Volkswagen, 和 Hyundai），再进行下一步：

```text
Step 2 -> Query 1:"Toyota 的电动汽车"
Step 2 -> Query 2:"Volkswagen 的电动汽车"
Step 2 -> Query 3:"Hyundai 的电动汽车"
```

### 2.2.4 查询路由

我们可以让大模型搜索不同数据源下的内容，比如，我们问了一个关于 HR 的问题，那么 LLM 就应该搜索公司的 HR 系统，但如果这个问题是关于客户数据，则它们应该在 CRM（Customer relationship management）系统中搜索。

## 2.3 RAG 评估

截至目前，RAG 评估指标分为以下几个维度：

- **检索质量**：是否找对文档？
- **重排序质量**：相关文档是否排在前面？
- **上下文质量**：检索内容是否足够且有用？
- **生成质量**：回答是否准确、可靠、切题？
- **引用质量**：引用是否充分且正确？
- **系统性能**：响应是否快速、稳定、经济？

我们在 1.3 节已经介绍了检索质量和重排序质量，下面我们来说后四个评估的维度

### 2.3.1 上下文质量评估

这是 RAG 特别重要的一组指标，用于评价提供给 LLM 的检索上下文，如下表所示：

| 指标                    | 中文名称     | 主要作用                     |
|-----------------------|----------|--------------------------|
| Context Precision     | 上下文精确率   | 相关 Chunk 是否排在无关 Chunk 前面 |
| Context Recall        | 上下文召回率   | 检索内容是否覆盖了回答所需的信息         |
| Context Relevance     | 上下文相关性   | 检索内容与用户问题有多相关            |
| Context Entity Recall | 上下文实体召回率 | 关键实体信息是否被检索到             |
| Noise Sensitivity     | 噪声敏感度    | 不相关信息是否会影响答案质量           |

特别注意，Ragas 的 **Context Precision** 不完全等同于传统的 `Precision@K`, 它通常考虑相关 Chunk 的排序位置，而 **Context Recall** 的一种实现则通过参考答案中的事实能否被检索上下文支持来计算。

### 2.3.2 生成质量评估

如下表所示，这部分衡量的是 LLM 基于检索信息生成答案的质量

| 指标                  | 中文名称  | 评估内容              |
|---------------------|-------|-------------------|
| Faithfulness        | 忠实度   | 回答中的事实是否得到检索上下文支持 |
| Answer Relevancy    | 答案相关性 | 回答是否切合用户的问题       |
| Answer Correctness  | 答案正确性 | 回答是否与标准答案一致       |
| Factual Correctness | 事实正确性 | 回答中的事实陈述与参考答案有多一致 |
| Semantic Similarity | 语义相似度 | 生成答案与参考答案的语义相似度   |
| Fluency             | 流畅性   | 语言是否自然、连贯         |
| Completeness        | 完整性   | 是否覆盖用户所需的全部关键信息   |

### 2.3.3 引用质量评估

| 指标                 | 中文名称 | 含义 |
|--------------------|---|---|
| Citation Recall    | 引用召回率 | 有多少事实性陈述得到了引用的充分支持 |
| Citation Precision | 引用精确率 | 有多少引用确定支持对应的事实性陈述 |

### 2.3.4 系统性能评估

| 指标         | 中文名称       | 含义                    |
|------------|------------|-----------------------|
| Latency    | 响应延迟       | 完成一次 RAG 请求所需的时间      |
| TTFT       | 首 Token 延迟 | 从请求开始到生成第一个 Token 的时间 |
| Throughput | 吞吐量        | 单位时间内系统处理的请求数量        |

另外，还有 Token Usage，Cost per Query 和 Error Rate 等指标，如果考虑到上线，还有 **P50,P95,P99 响应延迟** 等指标，它们衡量系统响应速度的百分位数指标，例如 `P50` 代表第 50 百分位延迟，这些指标比平均响应时间更能反映系统是否存在慢请求